## Data & AI 5/Artificial Intelligence: Machine Learning
## lifecycle Project
#### Team 30: Solovyova Aleksiya, Baldeva Vesela, Haazen Storm

### Step 2: Implementation and Pipeline
### Scope of This Step
This step focuses exclusively on:
- **Data preprocessing**: Handling missing values and data quality issues
- **Feature engineering**: Encoding categorical variables and selecting informative features
- **Pipeline implementation**: Creating a reproducible, robust data pipeline
- **Data leakage prevention**: Proper train-test splitting and transformation sequencing


#### Imports

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
import pandas as pd, numpy as np

## Loading and Inspection of the database

In this section we load the dataset and assign appropriate feature names. Additionally, missing values represented by '?' are replaced with NaN for easire handling later.

In [25]:
DATA_PATH = "data/agaricus-lepiota.data"
columns = [
        'class', 'cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor',
        'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color',
        'stalk-shape', 'stalk-root', 'stalk-surface-above-ring',
        'stalk-surface-below-ring', 'stalk-color-above-ring',
        'stalk-color-below-ring', 'veil-type', 'veil-color',
        'ring-number', 'ring-type', 'spore-print-color',
        'population', 'habitat'
    ]


df = pd.read_csv(DATA_PATH, names=columns)
df = df.replace('?', np.nan)

print(f"Loaded dataset with shape {df.shape}")
print(df.head())

Loaded dataset with shape (8124, 23)
  class cap-shape cap-surface cap-color bruises odor gill-attachment  \
0     p         x           s         n       t    p               f   
1     e         x           s         y       t    a               f   
2     e         b           s         w       t    l               f   
3     p         x           y         w       t    p               f   
4     e         x           s         g       f    n               f   

  gill-spacing gill-size gill-color  ... stalk-surface-below-ring  \
0            c         n          k  ...                        s   
1            c         b          k  ...                        s   
2            c         b          n  ...                        s   
3            c         n          n  ...                        s   
4            w         b          k  ...                        s   

  stalk-color-above-ring stalk-color-below-ring veil-type veil-color  \
0                      w                   

## Handling Missing Values in "stalk-root"

Two main options existed for addressing missing values in the "stalk-root" column:

**Option 1: Impute with most frequent values**
- Preserves the column and its predictive signal.
- Introduces noise and bias due to artificial data.

**Option 2: Drop the column**
- Avoids unreliable imputation and data leakage.
- Loses some predictive signal.

**Decision:** We dropped "stalk-root" because:
- Missing values stem from inconsistent data collection across sources, not the mushrooms themselves.
- Other Kaggle and academic analyses done on the dataset show that dropping this column actually improves model generalization.

In [26]:
if 'stalk-root' in df.columns:
    missing_rate = df['stalk-root'].isna().mean() * 100
    print(f"Dropping 'stalk-root' (missing {missing_rate:.1f}% of rows)")
    df = df.drop(columns=['stalk-root'])

Dropping 'stalk-root' (missing 30.5% of rows)


## Feature Preparation
The target variable is encoded as binary and all features are categorical, so no scaling is required.

In [27]:
y = df['class'].map({'e':0, 'p':1})
X = df.drop('class', axis=1)

## Pipeline Construction

We define the full preprocessing and modeling pipeline, which handles encoding, feature transformation/selection and model structure in one reproducible object.
- **OneHotEncoder** for categorical encoding.
- **SelectKBest** reduces dimensionality and selects the top 6 features after the encoding
- **Random Forest**:
  - Robust to overfitting
  - Handles non-linear relationships well
  - Good baseline model for classification
  - **Note**: This serves as a placeholder model for demonstration. In Step 3, different models will be compared using this same preprocessing pipeline.


In [28]:
categorical_features = X.columns.tolist()

cat_preprocess = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer([
    ('categorical', cat_preprocess, categorical_features)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('selector', SelectKBest(mutual_info_classif, k=6)),
    ('clf', RandomForestClassifier(random_state=42))
])

### Data Leakage Prevention
The pipeline is **fit only on training data**. This prevents:
- OneHotEncoder from learning categories from test data
- SelectKBest from selecting features based on test set patterns
- Any information from test set influencing the model

Although model selection and evaluation happen in later steps, we need to fit the pipeline now to:
1. **Validate** that all pipeline components work correctly
2. **Learn transformation parameters** (e.g., which categories exist, which features to select)
3. **Save a ready-to-use pipeline** for subsequent model comparison


In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

#Part-3-edit
# Going to export these to CSVs for use in step three:
y_train.to_frame('class').to_csv("./data/y_train.csv", index=False)
y_test.to_frame('class').to_csv("./data/y_test.csv", index=False)

X_train_pre = preprocessor.fit_transform(X_train)
X_test_pre = preprocessor.transform(X_test)

X_train_df = pd.DataFrame(X_train_pre.todense())
X_test_df = pd.DataFrame(X_test_pre.todense())

X_train_df.to_csv("./data/x_train.csv", index=False)
X_test_df.to_csv("./data/x_test.csv", index=False)

pipeline.fit(X_train, y_train)
print("Pipeline successfully fitted — ready for model training.")


Pipeline successfully fitted — ready for model training.


## Serialization and Persistence

The purpose of this step is for the saved pipeline to be used by team members for the next steps. The benefits include reproducibility, collaboration and if needed later, deployment readiness, as the same transformations will be applied to new data.

#### How to Use This Pipeline in Future Steps (this section may be removed after the model selection and evaluation)
To load and use this pipeline for model selection and evaluation:
```python
import joblib
pipeline = joblib.load("pipelines/mushroom_pipeline_step2.joblib")

# Access components
# pipeline.named_steps['preprocessor']
# pipeline.named_steps['selector']
# pipeline.named_steps['clf']  # Can be replaced with different models

# Use for predictions
# predictions = pipeline.predict(X_test)

In [30]:
import joblib, os
os.makedirs("pipelines", exist_ok=True)
joblib.dump(pipeline, "pipelines/mushroom_pipeline_step2.joblib")
print("Pipeline saved for later modeling.")

Pipeline saved for later modeling.
